# Nightly research Q02: how much does the circular "High-complexity Products" group move ECI?

**Question (QUEUE.md Q02):** Grouped ECI with vs. without the High-complexity aggregate, 2019–2023. Rank correlation, top movers, and the effect on the Spring 2026 report countries (Canada, Germany, Poland, Singapore, UAE, UK, Vietnam) and their peers.

- **Date:** 2026-11-12 (nightly agent)
- **Data:** `data/cleaned/<year>clean.csv` 2019–2023 (18 groups); `data/raw/2019–2023_mega.csv` (HS4 ECI benchmark and the chapter breakdown in A); `maitreemody/outputs/worldbank_gdp_per_capita_2019_2023.csv` (log GDPpc, PPP); peers from the "peer economies (…)" sentence of each `reports/2026-spring/*.pdf`
- **Years:** 2019 primary; 2020–2022 as checks; 2023 shown but suspect (D1)
- **Code:** `willbetts/nightly/lib/eci.py` (from Q01), not modified
- **Run time:** a few seconds (printed in the last code cell)

## TL;DR

- **Dropping the aggregate moves ECI moderately, not trivially.** Under the lab's rules (drop-any-zero, strict RCA, sign oriented), Spearman ρ between ECI with and without it is 0.921 in 2019 (0.932 / 0.966 / 0.939 / 0.935 in 2020–2023). The mean country moves 9.8 ranks out of 120 in 2019, 35.8% of countries move more than 10 ranks, and the largest move is 52 ranks (Section B).
- **Dropping it does *not* make the grouped ECI better.** Pre-stated secondary metrics: Spearman with log GDPpc falls in 4 of 5 years under the lab rules (mean change −0.037) and in 5 of 5 under the open rules (mean −0.062). Agreement with HS4 ECI barely changes under the lab rules (mean −0.004) and falls slightly under the open rules (mean −0.018) (Section C). The likely reason: **HS chapter 84 (machinery, 37.3% of the aggregate's chapters, 11.9% of all raw 2019 exports) is in no other group** (D4), so dropping the aggregate deletes machinery from the data (Section A).
- **Mechanism is as expected:** countries with RCA in the aggregate (24 of 120 in 2019) lose on average 4.6 ranks when it is dropped; the others gain 1.2 (Section D). The aggregate itself has the 2nd-highest PCI of the 18 groups in 2019 (1.716, after Transportation).
- **Report-level claims are fragile to this choice.** On 2019→2022 (oriented ECI), the *direction* of the ECI change flips for 3 of the 7 Spring 2026 countries (UAE, UK, Vietnam), and the "above/below peer average" call flips for 2 (Poland, Vietnam) (Section E). Big 2019 rank moves: Vietnam −15, Canada +15, Poland +18; Kuwait (a UAE peer) −52.
- **Side finding:** the notebook's sample includes the **`EU-28` aggregate row as a country** in every year (it ranks 10th–18th). Removing it gives Spearman 0.982–0.990 with the original and moves one country's ECI by up to 1.198 SD (Section G).

## Why this matters for the lab

This tests a piece of **our implementation**, not the theory. The research agenda (section 3) flags the High-complexity group as circular: it is defined *by* complexity and it double-counts trade that also sits in Chemicals, Chemical Manufacturing, Machinery and Construction, Transportation, Weapons and Services. If it moves ECI a lot, every report's ECI level, change, and peer comparison depends on an arbitrary data-cleaning choice, and the greedy optimizer (which maximizes this ECI) may be steering toward the aggregate's chapters for a mechanical reason. If dropping it *also* made the ECI track income better, it would be an easy fix to recommend. The result is "it matters, but dropping it alone is not the fix".

## Method

**Pre-stated primary metric** (written before running): for each year 2019–2023, compute ECI with all 18 groups (WITH) and with the aggregate column removed (WITHOUT) **on the same set of countries**, and report Spearman ρ, Pearson r, mean / median / max absolute rank change, the share of countries moving more than 10 ranks, and the mean |ΔECI| in SD units.

**Pre-stated secondary metric:** Spearman ρ of each version with (i) log GDP per capita (PPP, World Bank, ISO3 merge) and (ii) HS4 ECI from `data/raw` (the library's `load_hs4` + `eci_from_exports`, which reproduced Maitree's ~0.78 income correlation in Q01). "Better" means a higher ρ. **Baseline:** WITH (the current 18 groups).

**Specs (all reported):**
1. **lab (oriented):** the notebook's rules (drop any country with a zero in any of the 18 groups, strict RCA `>`), eigenvalues sorted, sign oriented to diversity. Zero filter applied on the 18-group matrix in both versions so the country set is identical. ISO3 keys; the `EU-28` row is excluded because it is not a country (see G). **Primary.**
2. **open (oriented):** keep zeros, RCA ≥ 1, total exports ≥ $1B (Julia/Maitree style).
3. **notebook as-is:** the exact notebook ECI (`eci.notebook_eci`, sign not fixed, names and `EU-28` as the notebook sees them). Reference only.

Spring 2026 peers are taken from the PDFs. "Report-style" claims are recomputed on 2019→2022 because the repo has no 2024 data (Q01 data request) and 2023 is suspect (D1).

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import eci
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
YEARS = [2019, 2020, 2021, 2022, 2023]
AGG = eci.AGGREGATE_GROUP
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
REPORT = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']
PEERS = {  # from the 'peer economies (...)' sentence of each reports/2026-spring/*.pdf
    'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
    'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'],
    'VNM': ['SGP', 'MYS', 'PHL']}
print('lib:', eci.__file__)

lib: /home/user/eci-research/willbetts/nightly/runs/2026-11-12_Q02-high-complexity-aggregate/../../lib/eci.py


In [2]:
# Cleaned 18-group data keyed on ISO3 (names mapped *before* notebook renames, D6). Row order is kept as in
# the CSV. Local patch: Comtrade code 381 ('Italy') is not in eci.M49_SPECIAL_ISO3, and pycountry's Italy is
# 380, so the library maps Italy to None. We add 381 -> ITA here (lib/ is not edited tonight).
N2C = eci._name_to_m49()
def iso3_of(name):
    c = N2C.get(str(name).strip())
    if c is None or c in eci.M49_AGGREGATES:
        return None
    return 'ITA' if c == 381 else eci.m49_to_iso3(c)

def load_iso(year):
    g = eci.load_groups(year, notebook_names=False)
    iso = pd.Series([iso3_of(n) for n in g.index], index=g.index)
    dropped = sorted(iso[iso.isna()].index)
    g = g.loc[iso.notna().values]
    g.index = iso.dropna().values
    assert not g.index.duplicated().any()
    return g, dropped

DATA, DROPPED = {}, {}
NB = {y: eci.load_groups(y) for y in YEARS}      # exactly what the notebook reads (names, incl. EU-28)
for y in YEARS:
    DATA[y], DROPPED[y] = load_iso(y)
print(pd.DataFrame({y: {'csv_rows': len(NB[y]), 'rows_with_iso3': len(DATA[y]),
                        'notebook_sample_after_zero_filter': len(eci.notebook_filter(NB[y])),
                        'lab_sample_after_zero_filter (ISO3)': len(eci.notebook_filter(DATA[y]))} for y in YEARS}).T)
print('rows without an ISO3 (dropped from the research samples):', DROPPED)

      csv_rows  rows_with_iso3  notebook_sample_after_zero_filter  lab_sample_after_zero_filter (ISO3)
2019       161             160                                121                                  120
2020       155             154                                121                                  120
2021       137             136                                106                                  105
2022       142             141                                121                                  120
2023       155             154                                123                                  122
rows without an ISO3 (dropped from the research samples): {2019: ['EU-28'], 2020: ['EU-28'], 2021: ['EU-28'], 2022: ['EU-28'], 2023: ['EU-28']}


## Results

### A. How big is the aggregate, and what is in it?

In [3]:
# A. How big is the aggregate, and how many countries have RCA in it?
rows = []
for y in YEARS:
    g = DATA[y]
    tot = g.to_numpy().sum()
    rest = g.drop(columns=[AGG]).to_numpy().sum()
    M = eci.notebook_mcp(eci.notebook_filter(g))
    rows.append({'year': y, 'total_all_18_$T': tot / 1e12, 'total_without_agg_$T': rest / 1e12,
                 'agg_share_of_total_%': 100 * g[AGG].sum() / tot,
                 'n_countries_filtered': len(M), 'n_with_M_agg=1': int(M[AGG].sum()),
                 'agg_ubiquity_rank_of_18 (1=least ubiquitous)': int(M.sum().rank(method='min').loc[AGG])})
A = pd.DataFrame(rows).set_index('year').round(3)
A

,total_all_18_$T,total_without_agg_$T,agg_share_of_total_%,n_countries_filtered,n_with_M_agg=1,agg_ubiquity_rank_of_18 (1=least ubiquitous)
year,,,,,,
2019,21.986,16.078,26.870,120,24,3
2020,20.462,14.979,26.797,120,25,4
2021,25.380,18.934,25.399,105,25,6
2022,27.413,20.535,25.089,120,26,4
2023,38.190,28.009,26.659,122,26,5


Takeaway: the aggregate is about a quarter of the summed cleaned exports every year (25.1%–26.9%), but only 24–26 countries have RCA in it, the 3rd–6th least ubiquitous of the 18 groups.

In [4]:
# PCI of every group in the 18-group spec (2019, lab filter, oriented). Where does the aggregate sit?
e19, p19, d19 = eci.eci_pci(eci.notebook_mcp(eci.notebook_filter(DATA[2019])))
pci_tab = p19.sort_values(ascending=False).round(3).to_frame('PCI_2019')
pci_tab['ubiquity'] = eci.notebook_mcp(eci.notebook_filter(DATA[2019])).sum().reindex(pci_tab.index).astype(int)
pci_tab

,PCI_2019,ubiquity
Transportation,1.872,20
High-complexity Products,1.716,24
Daily Instruments,1.065,35
Machinery and Construction,1.055,24
Raw Manufacturing,0.649,35
Chemicals,0.526,29
Chemical Manufacturing,0.356,39
Services and Utilities,0.205,19
Weapons,0.165,26
Clothing and Accessories,-0.284,39


Takeaway: in 2019 the aggregate has the 2nd-highest PCI (1.716), right behind Transportation (1.872), so it acts as a strong "high-complexity" marker in the eigenvector.

In [5]:
# What is inside the aggregate? World exports by HS chapter from data/raw (2019, all reporters except the
# M49 regional aggregates), for the chapters methodology.md lists for 'High-complexity Products', and
# whether each chapter ALSO belongs to a regular group (methodology.md Industry groups table).
OTHER_GROUP = {'28': 'Chemicals', '29': 'Chemicals', '30': 'Chemical Manufacturing', '38': 'Chemical Manufacturing',
               '83': 'Machinery and Construction', '84': None, '87': 'Transportation', '92': 'Weapons', '99': 'Services and Utilities'}
raw = pd.read_csv(eci.REPO / 'data/raw/2019_mega.csv', index_col=0)
ch = raw.pop('commodity_code').astype(str).str.zfill(6).str[:2]
raw = raw.drop(columns=[c for c in raw.columns if int(c) in eci.M49_AGGREGATES]).apply(pd.to_numeric, errors='coerce').fillna(0)
by_ch = raw.sum(axis=1).groupby(ch.values).sum()
agg_ch = by_ch.reindex(list(OTHER_GROUP)).fillna(0)
tab = pd.DataFrame({'world_exports_2019_$B': agg_ch / 1e9, 'share_of_aggregate_%': 100 * agg_ch / agg_ch.sum(),
                    'also_in_group': pd.Series(OTHER_GROUP)})
print(f"aggregate chapters = {100 * agg_ch.sum() / by_ch.sum():.1f}% of all raw 2019 exports; "
      f"chapter 84 alone = {100 * by_ch['84'] / by_ch.sum():.1f}%")
tab.round(1)

aggregate chapters = 31.9% of all raw 2019 exports; chapter 84 alone = 11.9%


,world_exports_2019_$B,share_of_aggregate_%,also_in_group
28,118.8,2.0,Chemicals
29,393.7,6.7,Chemicals
30,587.4,10.0,Chemical Manufacturing
38,200.6,3.4,Chemical Manufacturing
83,71.4,1.2,Machinery and Construction
84,2192.3,37.3,NaN
87,1472.7,25.1,Transportation
92,6.8,0.1,Weapons
99,833.5,14.2,Services and Utilities


Takeaway: chapter 84 (machinery) is the largest piece of the aggregate (37.3%) and belongs to **no** regular group (KNOWN-ISSUES D4). Every other chapter is double-counted with a regular group. So "drop the aggregate" both removes the double counting and silently removes machinery. The raw chapter share (31.9%) differs from the cleaned share in the table above (26.9%) because the cleaned files cover a different reporter set; it is not reconciled here (that is Q06's job).

### B. With vs. without, same countries, three specs

In [6]:
# B. Pre-stated comparison. Each spec computes ECI with all 18 groups (WITH) and with the
# aggregate dropped (WITHOUT) on the SAME country set, then compares.
def spec_lab(X):        # notebook rules (drop-any-zero, strict RCA), sign oriented to diversity
    return eci.eci_pci(eci.notebook_mcp(eci.notebook_filter(X)))
def spec_open(X):       # Julia/Maitree style: keep zeros, RCA >= 1, $1B minimum exports
    X = X.loc[X.sum(axis=1) >= 1e9]
    return eci.eci_pci(eci.mcp(X, strict=False))
def spec_asis(X):       # notebook exactly as it runs (sign NOT fixed) -- reference only
    s = eci.notebook_eci(X)
    return s, None, {}
SPECS = {'lab (oriented)': spec_lab, 'open (oriented)': spec_open, 'notebook as-is': spec_asis}

def with_without(year, spec):
    X = NB[year] if spec is spec_asis else DATA[year]
    if spec is spec_lab or spec is spec_asis:
        # the zero filter is applied to the 18-group matrix in both cases so the set is the same
        X = eci.notebook_filter(X)
    ew, _, dw = spec(X)
    eo, _, do = spec(X.drop(columns=[AGG]))
    common = ew.index.intersection(eo.index)
    return ew.loc[common], eo.loc[common], dw, do, len(ew), len(eo)

def compare(ew, eo):
    rw, ro = ew.rank(ascending=False), eo.rank(ascending=False)
    dr = (ro - rw).abs()
    return {'n': len(ew), 'spearman': spearmanr(ew, eo)[0], 'pearson': pearsonr(ew, eo)[0],
            'mean_abs_rank_change': dr.mean(), 'median_abs_rank_change': dr.median(),
            'max_abs_rank_change': dr.max(), 'share_moving_>10_ranks_%': 100 * (dr > 10).mean(),
            'mean_abs_dECI_sd': (eo - ew).abs().mean()}

In [7]:
RES, ECIS = [], {}
for name, f in SPECS.items():
    for y in YEARS:
        ew, eo, dw, do, nw, no = with_without(y, f)
        ECIS[(name, y)] = (ew, eo)
        r = {'spec': name, 'year': y, **compare(ew, eo)}
        if dw:
            r.update({'eigengap_with': dw['eigengap_23'], 'eigengap_without': do['eigengap_23']})
        RES.append(r)
B = pd.DataFrame(RES).set_index(['spec', 'year'])
B.round(3)

n  spearman  pearson  mean_abs_rank_change  median_abs_rank_change  max_abs_rank_change  share_moving_>10_ranks_%  mean_abs_dECI_sd  \
spec            year                                                                                                                                          
lab (oriented)  2019  120     0.921    0.922                 9.825                    7.00                 52.0                    35.833             0.284   
                2020  120     0.932    0.926                 8.425                    5.00                 59.0                    25.000             0.250   
                2021  105     0.966    0.964                 5.762                    4.00                 28.0                    11.429             0.212   
                2022  120     0.939    0.927                 7.975                    5.00                 57.0                    27.500             0.284   
                2023  122     0.935    0.936                 8.123                    5.00                 55.0                    23.770             0.239   
open (oriented) 2019  129     0.924    0.908                10.209                    6.00                 50.0                    37.984             0.288   
                2020  127     0.916    0.888                 9.756                    6.00                 77.0                    30.709             0.299   
                2021  120     0.950    0.952                 8.150                    6.25                 44.0                    29.167             0.243   
                2022  120     0.961    0.942                 7.050                    6.00                 45.0                    20.000             0.254   
                2023  130     0.917    0.892                10.823                    7.00                 62.0                    36.923             0.322   
notebook as-is  2019  121     0.908    0.916                10.223                    7.00                 63.0                    30.579             0.288   
                2020  121     0.934    0.920                 8.380                    6.00                 65.0                    21.488             0.261   
                2021  106     0.963    0.962                 6.377                    5.00                 27.0                    16.981             0.214   
                2022  121     0.949    0.951                 7.463                    5.00                 46.0                    23.140             0.232   
                2023  123     0.934    0.936                 8.463                    5.00                 48.0                    27.642             0.241   

                      eigengap_with  eigengap_without  
spec            year                                   
lab (oriented)  2019          0.074             0.083  
                2020          0.082             0.082  
                2021          0.067             0.068  
                2022          0.081             0.068  
                2023          0.113             0.088  
open (oriented) 2019          0.069             0.081  
                2020          0.092             0.083  
                2021          0.062             0.079  
                2022          0.089             0.063  
                2023          0.101             0.078  
notebook as-is  2019            NaN               NaN  
                2020            NaN               NaN  
                2021            NaN               NaN  
                2022            NaN               NaN  
                2023            NaN               NaN

Takeaway: under the primary lab spec, ρ is 0.921–0.966 across years, the mean rank move is 5.8–9.8 places, and the max move is 28–59 places. The open spec and the as-is notebook give the same picture (ρ 0.908–0.966). The eigengap λ2−λ3 is small either way (0.062–0.113 across both oriented specs), and dropping the aggregate does not consistently widen it (it narrows in 2022 and 2023 under the lab spec).

In [8]:
ew, eo = ECIS[('lab (oriented)', 2019)]
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(ew, eo, s=14, alpha=.6, color='#3b6ea8')
for c in REPORT:
    if c in ew.index:
        ax.scatter(ew[c], eo[c], s=40, color='#c0392b'); ax.annotate(c, (ew[c], eo[c]), fontsize=8, xytext=(3, 3), textcoords='offset points')
lim = [min(ew.min(), eo.min()) - .2, max(ew.max(), eo.max()) + .2]
ax.plot(lim, lim, color='grey', lw=.8, ls='--')
ax.set_xlabel('ECI with High-complexity group (18 groups)'); ax.set_ylabel('ECI without it (17 groups)')
ax.set_title(f'2019, lab rules, sign oriented (n={len(ew)}, Spearman {spearmanr(ew, eo)[0]:.3f})', fontsize=9)
fig.tight_layout(); fig.savefig(FIG / 'with_vs_without_2019.png', dpi=110); plt.show()

Takeaway: most countries stay near the diagonal; a visible minority, mostly low-diversity countries, move far.

### C. Does dropping the aggregate make the grouped ECI better? (pre-stated secondary metric)

In [9]:
# C. Secondary metric (pre-stated): which version agrees better with income and with HS4 ECI?
gdp = pd.read_csv(eci.REPO / 'maitreemody/outputs/worldbank_gdp_per_capita_2019_2023.csv')
lg = gdp.dropna(subset=['gdppc_ppp']).assign(l=lambda d: np.log(d.gdppc_ppp)).set_index(['iso3', 'year'])['l']
HS4 = {}
for y in YEARS:
    X4 = eci.load_hs4(y)
    X4.index = [eci.m49_to_iso3(c) for c in X4.index]
    X4 = X4.loc[[i is not None for i in X4.index]]
    X4 = X4.groupby(level=0).sum()
    HS4[y] = eci.eci_from_exports(X4)
rows = []
for name in ['lab (oriented)', 'open (oriented)']:
    for y in YEARS:
        ew, eo = ECIS[(name, y)]
        inc = lg.xs(y, level='year')
        c1 = ew.index.intersection(inc.index)
        c2 = ew.index.intersection(HS4[y].index)
        rows.append({'spec': name, 'year': y, 'n_income': len(c1),
                     'rho_lnGDPpc_with': spearmanr(ew[c1], inc[c1])[0], 'rho_lnGDPpc_without': spearmanr(eo[c1], inc[c1])[0],
                     'n_hs4': len(c2),
                     'rho_HS4ECI_with': spearmanr(ew[c2], HS4[y][c2])[0], 'rho_HS4ECI_without': spearmanr(eo[c2], HS4[y][c2])[0]})
C = pd.DataFrame(rows).set_index(['spec', 'year'])
C['d_income'] = C.rho_lnGDPpc_without - C.rho_lnGDPpc_with
C['d_hs4'] = C.rho_HS4ECI_without - C.rho_HS4ECI_with
C.round(3)

n_income  rho_lnGDPpc_with  rho_lnGDPpc_without  n_hs4  rho_HS4ECI_with  rho_HS4ECI_without  d_income  d_hs4
spec            year                                                                                                              
lab (oriented)  2019       118             0.480                0.471    105            0.727               0.730    -0.008  0.003
                2020       117             0.534                0.505    105            0.771               0.765    -0.029 -0.007
                2021       104             0.527                0.545     94            0.806               0.850     0.018  0.043
                2022       118             0.555                0.467    105            0.760               0.719    -0.088 -0.041
                2023       121             0.513                0.437    109            0.724               0.704    -0.077 -0.020
open (oriented) 2019       128             0.420                0.378    127            0.714               0.694    -0.042 -0.021
                2020       126             0.499                0.404    125            0.727               0.684    -0.095 -0.043
                2021       119             0.541                0.515    119            0.828               0.828    -0.026 -0.001
                2022       119             0.493                0.444    118            0.743               0.729    -0.049 -0.014
                2023       129             0.515                0.417    129            0.716               0.706    -0.098 -0.010

In [10]:
C.groupby(level='spec')[['d_income', 'd_hs4']].agg(['mean', 'min', 'max', lambda s: int((s > 0).sum())]).round(3).rename(columns={'<lambda_0>': 'years_without_better'})

d_income                                     d_hs4                                   
                    mean    min    max years_without_better   mean    min    max years_without_better
spec                                                                                                 
lab (oriented)    -0.037 -0.088  0.018                    1 -0.004 -0.041  0.043                    2
open (oriented)   -0.062 -0.098 -0.026                    0 -0.018 -0.043 -0.001                    0

Takeaway: no. Without the aggregate, the income correlation is lower in 4/5 (lab) and 5/5 (open) years, and the HS4 agreement is roughly flat (lab, mean −0.004, better in 2/5 years) or slightly lower (open, mean −0.018, better in 0/5). 2021, the year with the smallest lab sample (105), is the only year where the lab spec improves on both. Because chapter 84 vanishes when the aggregate is dropped (A), this is **not** a test of "aggregate removed with machinery kept"; that needs the D4 remap (Q05).

### D. Who moves, and why?

In [11]:
# D. Top movers, 2019, lab spec (oriented). Positive dRank = moves UP without the aggregate.
ew, eo = ECIS[('lab (oriented)', 2019)]
M19 = eci.notebook_mcp(eci.notebook_filter(DATA[2019]))
mv = pd.DataFrame({'eci_with': ew, 'eci_without': eo, 'rank_with': ew.rank(ascending=False), 'rank_without': eo.rank(ascending=False)})
mv['dRank_up'] = mv.rank_with - mv.rank_without
mv['M_agg'] = M19.loc[mv.index, AGG].astype(int)
mv['diversity_18'] = M19.loc[mv.index].sum(axis=1).astype(int)
top = pd.concat([mv.sort_values('dRank_up').head(8), mv.sort_values('dRank_up').tail(8)])
top.round(3)

,eci_with,eci_without,rank_with,rank_without,dRank_up,M_agg,diversity_18
BRN,-0.160,-1.893,64.5,116.5,-52.0,0,2
KWT,-0.160,-1.893,64.5,116.5,-52.0,0,2
JAM,0.242,-0.627,45.0,85.0,-40.0,1,5
PYF,-0.359,-1.261,74.0,111.5,-37.5,0,5
COG,0.230,-0.571,46.0,80.0,-34.0,0,3
BHR,-0.411,-1.075,77.0,105.0,-28.0,0,4
GUY,-0.773,-1.261,86.0,111.5,-25.5,0,6
MDA,0.871,0.180,27.0,51.0,-24.0,0,5
LKA,-0.364,-0.109,75.0,61.0,14.0,0,6
BEN,-0.789,-0.363,87.0,72.0,15.0,0,5


Takeaway: the biggest losers in 2019 are low-diversity countries (Brunei and Kuwait, tied with diversity 2; Jamaica; French Polynesia; Congo; Bahrain); the biggest gainers include Russia (+37), Croatia (+22), Mozambique (+22), Luxembourg (+21), and Poland (+18). Most top movers do **not** have RCA in the aggregate, so the moves are about the eigenvector re-sorting, not only about losing a "complex" column.

In [12]:
# Mechanism check: do countries WITH RCA in the aggregate lose rank when it is dropped?
mech = []
for y in YEARS:
    ew, eo = ECIS[('lab (oriented)', y)]
    M = eci.notebook_mcp(eci.notebook_filter(DATA[y])).loc[ew.index]
    dr = ew.rank(ascending=False) - eo.rank(ascending=False)
    mech.append({'year': y, 'mean_dRank_up_M_agg=1': dr[M[AGG] == 1].mean(), 'mean_dRank_up_M_agg=0': dr[M[AGG] == 0].mean(),
                 'n_M_agg=1': int((M[AGG] == 1).sum()), 'mean_dECI_M_agg=1': (eo - ew)[M[AGG] == 1].mean(), 'mean_dECI_M_agg=0': (eo - ew)[M[AGG] == 0].mean()})
pd.DataFrame(mech).set_index('year').round(3)

,mean_dRank_up_M_agg=1,mean_dRank_up_M_agg=0,n_M_agg=1,mean_dECI_M_agg=1,mean_dECI_M_agg=0
year,,,,,
2019,-4.604,1.151,24,-0.153,0.038
2020,-7.600,2.000,25,-0.260,0.068
2021,-3.760,1.175,25,-0.150,0.047
2022,-8.538,2.362,26,-0.366,0.101
2023,-7.115,1.927,26,-0.237,0.064


Takeaway: countries with RCA in the aggregate lose 3.8–8.5 ranks on average when it is dropped, and the rest gain 1.2–2.4, in every year. That is the expected direction: the aggregate was giving its holders a complexity boost.

### E. Spring 2026 report countries and their peers

In [13]:
# E. Spring 2026 report countries and their peers (lab spec, oriented). Ranks are out of n that year.
focus = sorted(set(REPORT) | {p for v in PEERS.values() for p in v})
rows = []
for y in YEARS:
    ew, eo = ECIS[('lab (oriented)', y)]
    rw, ro = ew.rank(ascending=False), eo.rank(ascending=False)
    for c in focus:
        rows.append({'iso3': c, 'year': y, 'in_sample': c in ew.index,
                     'eci_with': ew.get(c, np.nan), 'eci_without': eo.get(c, np.nan),
                     'rank_with': rw.get(c, np.nan), 'rank_without': ro.get(c, np.nan), 'n': len(ew)})
E = pd.DataFrame(rows)
E19 = E[E.year == 2019].set_index('iso3').drop(columns='year')
E19['dRank_up'] = E19.rank_with - E19.rank_without
E19.round(3)

,in_sample,eci_with,eci_without,rank_with,rank_without,n,dRank_up
iso3,,,,,,,
ARE,True,-2.051,-1.914,119.0,119.0,120,0.0
AUS,True,-1.169,-0.960,110.0,99.0,120,11.0
CAN,True,-1.058,-0.798,108.0,93.0,120,15.0
CZE,True,1.881,2.047,6.0,2.0,120,4.0
DEU,True,2.031,1.928,1.0,7.0,120,-6.0
FRA,True,0.986,0.746,23.0,31.0,120,-8.0
GBR,True,0.461,0.504,39.0,34.0,120,5.0
HUN,True,1.941,1.999,4.0,3.0,120,1.0
ITA,True,1.137,1.113,17.0,17.0,120,0.0


Takeaway (2019): Germany drops from rank 1 to 7 and Saudi Arabia from 2 to 9; Poland rises 36 → 18 and Canada 108 → 93; Vietnam falls 34 → 49. Kuwait, a UAE peer, falls 64.5 → 116.5. The Philippines and Qatar are not in the 2019 sample at all (zero in some group).

In [14]:
# Coverage: in how many of the 5 years is each focus country in the lab sample (drop-any-zero filter)?
E.groupby('iso3').in_sample.sum().astype(int).to_frame('years_in_sample').T

iso3,ARE,AUS,CAN,CZE,DEU,FRA,GBR,HUN,ITA,JPN,KOR,KWT,MYS,PHL,POL,QAT,SAU,SGP,USA,VNM
years_in_sample,5,5,5,5,5,5,5,5,5,5,5,4,5,2,5,1,4,5,5,5


Takeaway: the Philippines (2 of 5 years), Qatar (1 of 5), Kuwait and Saudi Arabia (4 of 5) drop out of the lab sample in some years because of the zero filter (D2), so the UAE and Vietnam peer averages below rest on 1–2 peers.

In [15]:
# Do the report-style claims change? 2019 -> 2022 ECI change (2023 suspect, D1) and the change vs. the
# peer-average change ('above/below peer average'), with vs without the aggregate.
def val(c, y, col):
    r = E[(E.iso3 == c) & (E.year == y)]
    return float(r[col].iloc[0]) if len(r) else np.nan
out = []
for c in REPORT:
    r = {'country': c}
    for tag, col in [('with', 'eci_with'), ('without', 'eci_without')]:
        ch = val(c, 2022, col) - val(c, 2019, col)
        pchs = [val(p, 2022, col) - val(p, 2019, col) for p in PEERS[c]]
        pch = np.nanmean(pchs)
        r['peers_used'] = ','.join(p for p, v in zip(PEERS[c], pchs) if not np.isnan(v))
        r[f'chg_{tag}'] = ch
        r[f'dir_{tag}'] = '' if np.isnan(ch) else ('up' if ch > 0 else 'down')
        r[f'peer_avg_chg_{tag}'] = pch
        r[f'vs_peers_{tag}'] = '' if np.isnan(ch) or np.isnan(pch) else ('above' if ch > pch else 'below')
    r['direction_flips'] = r['dir_with'] != r['dir_without']
    r['vs_peers_flips'] = r['vs_peers_with'] != r['vs_peers_without']
    out.append(r)
EC = pd.DataFrame(out).set_index('country')
EC.round(3)

,peers_used,chg_with,dir_with,peer_avg_chg_with,vs_peers_with,chg_without,dir_without,peer_avg_chg_without,vs_peers_without,direction_flips,vs_peers_flips
country,,,,,,,,,,,
CAN,"USA,AUS,GBR",0.571,up,0.252,above,0.340,up,-0.156,above,False,False
DEU,"FRA,GBR,ITA",-0.388,down,0.088,below,-0.339,down,-0.172,below,False,False
POL,"DEU,CZE,HUN",-0.229,down,-0.207,below,-0.268,down,-0.371,above,False,True
SGP,"KOR,JPN,DEU",0.797,up,-0.014,above,0.564,up,-0.187,above,False,False
ARE,KWT,0.194,up,-1.865,above,-0.280,down,-0.702,above,True,False
GBR,"DEU,FRA,ITA",1.029,up,-0.385,above,-0.120,down,-0.245,above,True,False
VNM,"SGP,MYS",-0.213,down,0.326,below,0.558,up,-0.093,above,True,True


Takeaway: the 2019→2022 direction flips for UAE (up → down), UK (up → down) and Vietnam (down → up); the vs-peer-average call flips for Poland (below → above) and Vietnam (below → above). Canada, Germany and Singapore are unchanged on both. These are oriented ECIs on 2019→2022, not the published 2019→2024 numbers, so they show the *sensitivity* of the claims, not corrections to the reports.

In [16]:
fig, axes = plt.subplots(1, 7, figsize=(14, 2.8), sharey=True)
for ax, c in zip(axes, REPORT):
    d = E[E.iso3 == c].set_index('year')
    ax.plot(d.index, d.eci_with, marker='o', ms=3, label='with aggregate (18)', color='#3b6ea8')
    ax.plot(d.index, d.eci_without, marker='s', ms=3, label='without (17)', color='#e08e2b')
    ax.set_title(c, fontsize=9); ax.set_xticks(YEARS); ax.set_xticklabels([str(y)[2:] for y in YEARS], fontsize=7)
    ax.axvspan(2022.5, 2023.5, color='grey', alpha=.12)
axes[0].set_ylabel('ECI (lab rules, oriented)'); axes[0].legend(fontsize=6, loc='lower left')
fig.suptitle('Spring 2026 report countries: grouped ECI with vs without the High-complexity aggregate (2023 shaded: D1)', fontsize=9)
fig.tight_layout(); fig.savefig(FIG / 'spring2026_with_without.png', dpi=100); plt.show()

Takeaway: the two lines track each other for Germany and Singapore but diverge in level or slope for UK, UAE and Vietnam.

### F. Notebook as-is: does dropping the aggregate add sign flips?

In [17]:
# F. Notebook as-is: how often does dropping the aggregate also flip the notebook's arbitrary sign?
asis = []
for y in YEARS:
    ew, eo = ECIS[('notebook as-is', y)]
    X = eci.notebook_filter(NB[y]); kc_w = eci.notebook_mcp(X).sum(1); kc_o = eci.notebook_mcp(X.drop(columns=[AGG])).sum(1)
    asis.append({'year': y, 'sign_ok_with': bool(np.corrcoef(ew, kc_w.loc[ew.index])[0, 1] > 0),
                 'sign_ok_without': bool(np.corrcoef(eo, kc_o.loc[eo.index])[0, 1] > 0), 'pearson_with_vs_without': pearsonr(ew, eo)[0]})
pd.DataFrame(asis).set_index('year').round(3)

,sign_ok_with,sign_ok_without,pearson_with_vs_without
year,,,
2019,True,True,0.916
2020,True,True,0.920
2021,False,False,0.962
2022,False,False,0.951
2023,False,False,0.936


Takeaway: no. The notebook's sign is wrong in 2021–2023 both with and without the aggregate (matching Q01) and right in 2019–2020 both ways, so this choice does not add sign flips on these five matrices.

### G. Side finding: `EU-28` is in the notebook's country sample

In [18]:
# G. Side finding: the notebook's sample includes the 'EU-28' aggregate row as if it were a country.
# How much does removing it move the notebook-rule ECI (oriented, 18 groups)?
eu = []
for y in YEARS:
    X = eci.notebook_filter(NB[y])
    a, _, _ = eci.eci_pci(eci.notebook_mcp(X))
    b, _, _ = eci.eci_pci(eci.notebook_mcp(X.drop(index='EU-28')))
    a = a.drop('EU-28')
    eu.append({'year': y, 'EU-28_in_notebook_sample': True, 'EU-28_ECI': float(eci.eci_pci(eci.notebook_mcp(X))[0]['EU-28']),
               'EU-28_rank': int(eci.eci_pci(eci.notebook_mcp(X))[0].rank(ascending=False)['EU-28']),
               'n': len(X), 'spearman_with_vs_without_EU28': spearmanr(a, b.loc[a.index])[0],
               'max_abs_dECI': (a - b.loc[a.index]).abs().max()})
pd.DataFrame(eu).set_index('year').round(4)

,EU-28_in_notebook_sample,EU-28_ECI,EU-28_rank,n,spearman_with_vs_without_EU28,max_abs_dECI
year,,,,,,
2019,True,1.6595,10,121,0.9820,1.1980
2020,True,1.5732,10,121,0.9840,1.1222
2021,True,1.3504,12,106,0.9901,0.8206
2022,True,1.2414,18,121,0.9904,0.8737
2023,True,1.2347,12,123,0.9841,0.8890


Takeaway: the `EU-28` row (M49 code 97, a regional aggregate) passes the zero filter every year and gets a high ECI (rank 10–18). Removing it leaves ranks very similar (ρ 0.982–0.990), but individual ECIs move by up to 0.82–1.20 SD. Its exports are also added on top of its members' own rows in the world totals used for RCA. It should be excluded in the notebook.

In [19]:
B.round(4).to_csv(FIG / 'with_without_summary.csv')
E.round(4).to_csv(FIG / 'spring2026_focus_eci.csv', index=False)
print(f'run time: {time.time() - T0:.1f} s')

run time: 3.4 s


## Caveats

- **Confounded with D4.** Dropping the aggregate also drops HS 84 entirely, so Section C cannot separate "removing circularity" from "losing machinery". Q05 (remap 84/85) should be done before deciding.
- **Small samples, few columns.** 105–130 countries and 17–18 columns. Grouped ECI has ties (e.g. Brunei and Kuwait share identical rows in 2019), so single rank moves of tied countries are not meaningful.
- **Peer comparisons are thin.** UAE's 2019→2022 peer average uses only Kuwait, and Vietnam's uses Singapore and Malaysia, because of the zero filter (D2).
- **Not the published report numbers.** The Spring 2026 reports used 2019–2024 data that are not in the repo (Q01 data request), so Section E shows how sensitive the *type* of claim is, not what the reports should have said.
- **Nominal values, per-year samples.** Exports are nominal US$, and each year is standardized over its own country set (D2/D3). 2023 is shown but suspect (D1).
- **Income and HS4 comparisons are cross-sectional correlations**, not tests of growth prediction.

## Helpful to know

- **`eci.m49_to_iso3(381)` returns `None`.** Comtrade codes Italy as 381, pycountry's Italy is 380, and 381 is missing from `eci.M49_SPECIAL_ISO3`. Any ISO3-keyed analysis using the library silently drops Italy. This notebook patches it locally (`381 → ITA`); the library should get the same one-line fix (not done here, per protocol).
- **`EU-28` is a row in every cleaned CSV and survives the notebook's zero filter** (Section G). `eci.group_name_to_iso3` maps it to `None`, so ISO3-keyed work drops it automatically, but the notebook and anything keyed on names keeps it.
- **Row order matters for the as-is notebook sign.** Re-indexing the cleaned data (e.g. sorting by ISO3) changes which country is row 0 and can flip the notebook's sign. Use `eci.load_groups(y)` in CSV order when reproducing the notebook.
- After dropping the aggregate, the 17 remaining groups no longer double count (each chapter is in at most one group, per the methodology table), but HS 84 is then in none.
- Grouped-ECI ties: round before ranking (Q01), and use `rank(method='average')` as here so tied countries get the same rank.

## New research avenues (each about one night)

1. **Aggregate removed, machinery kept:** rebuild the groups from `data/raw` with HS 84 added to *Machinery and Construction* and the aggregate dropped, then rerun Section C. This is the clean test of circularity (overlaps Q05; could be its first slice).
2. **Greedy target with vs. without the aggregate:** for each Spring 2026 country, which group gives the largest one-step ECI gain from a 5% export bump with and without the aggregate (no full greedy run). Shows whether the optimizer is chasing the aggregate mechanically (complements Q11).
3. **EU-28 and other non-country rows:** list every row in the cleaned CSVs that is not a country (EU-28 and any others), and measure their combined effect on RCA world shares and ECI.
4. **Tie structure:** count identical M rows per year with and without the aggregate. Dropping a column should create more ties; quantify how many countries become indistinguishable (folds in Q01 avenue 4).

## Data issues / data needed

- **`EU-28` is included as a country in the notebook's ECI** (Section G). This is a notebook/cleaning issue. Suggested fix: drop rows whose Comtrade code is in the regional-aggregate list before `df_to_matrix`. Suggest adding it to KNOWN-ISSUES (e.g. D7).
- **HS 84 lives only in the aggregate** (D4, confirmed here at 37.3% of the aggregate's 2019 chapters). Any decision to drop the aggregate must come with the D4 remap.
- **Library gap:** `M49_SPECIAL_ISO3` lacks `381: 'ITA'`.
- No external data needed tonight. Details: `willbetts/nightly/data-requests/2026-11-12-eu28-and-italy-codes.md`.